# GlassDesign ML — самостоятельный запуск в Google Colab
Бесплатная воспроизводимая демонстрация оригинальных моделей и веб-интерфейса. **Это не постоянно работающий сайт.** При запуске блокнота создаётся временная ссылка, действующая только пока работает среда Colab. Запустить можно в любой момент без участия автора.

Для запуска нужны: (1) открытый на чтение репозиторий `polinashidlovskaya/glassdesign-ml`; (2) ссылка с доступом «Все, у кого есть ссылка → Читатель» к архиву `GlassDesignML_models.zip` (1,2 ГБ), который содержит оригинальные четыре `.joblib`. **Никогда не загружайте файл с чужими или непроверенными моделями** — формат joblib/pickle может выполнять произвольный код.

**Шаги:** вставьте ссылку в ячейку ниже → «Среда выполнения → Выполнить все» → разрешите выполнение и дождитесь вывода временной ссылки. Веб-интерфейс затем доступен в браузере. После остановки Colab ссылку нужно получить заново.

In [ ]:
# ВСТАВЬТЕ ПУБЛИЧНУЮ ССЫЛКУ НА АРХИВ МОДЕЛЕЙ ИЗ GOOGLE DRIVE
MODEL_ARCHIVE_LINK = ""
assert MODEL_ARCHIVE_LINK.startswith('https://'), 'Укажите ссылку на архив моделей с доступом по ссылке'


In [ ]:
import re, sys, shutil, pathlib, subprocess, zipfile
project=pathlib.Path('/content/glassdesign-ml')
if not project.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/polinashidlovskaya/glassdesign-ml.git',str(project)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(project/'requirements.txt'),'gdown'],check=True)
import gdown
match = re.search(r'/file/d/([^/?]+)', MODEL_ARCHIVE_LINK) or re.search(r'[?&]id=([^&]+)', MODEL_ARCHIVE_LINK)
if not match:
    raise ValueError('Используйте ссылку на файл Google Drive формата https://drive.google.com/file/d/ID/view')
archive=pathlib.Path('/content/glassdesign_models.zip')
if not archive.exists() or archive.stat().st_size < 1_000_000_000:
    gdown.download(id=match.group(1), output=str(archive), quiet=False)
if not zipfile.is_zipfile(archive):
    raise RuntimeError('Не удалось загрузить ZIP. Проверьте доступ «Все, у кого есть ссылка»')
expected=['density_extra_trees_final.joblib','refractive_index_extra_trees_final_v2.joblib','rf_tg_final.joblib','extra_trees_softening_final.joblib']
dest=project/'runtime'/'GlassDesignML'/'models'
dest.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for filename in expected:
        source=f'models/{filename}'
        if source not in z.namelist():
            raise RuntimeError('В архиве отсутствует '+source)
        target=dest/filename
        if not target.is_file():
            with z.open(source) as src, target.open('wb') as dst:
                shutil.copyfileobj(src,dst,1024*1024)
        print('✅',filename,round(target.stat().st_size/1024**2,1),'МиБ')
assert all((dest/n).is_file() for n in expected)
print('✅ Четыре модели готовы к запуску')


In [ ]:
import os,time,subprocess,pathlib,re,requests
project=pathlib.Path('/content/glassdesign-ml')
port='8501'
app_log=open('/content/glassdesign_streamlit.log','w')
app=subprocess.Popen(['streamlit','run','streamlit_app.py','--server.port='+port,'--server.address=0.0.0.0','--server.headless=true','--browser.gatherUsageStats=false'],cwd=str(project),stdout=app_log,stderr=subprocess.STDOUT)
for _ in range(60):
    if app.poll() is not None:
        raise RuntimeError('Streamlit завершился. Откройте /content/glassdesign_streamlit.log')
    try:
        if requests.get('http://127.0.0.1:'+port+'/_stcore/health',timeout=1).ok:break
    except requests.RequestException:pass
    time.sleep(1)
else:raise RuntimeError('Streamlit не запустился за 60 секунд')
cf=pathlib.Path('/content/cloudflared')
if not cf.exists():
    data=requests.get('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',timeout=120)
    data.raise_for_status(); cf.write_bytes(data.content); cf.chmod(0o755)
tunnel_log=pathlib.Path('/content/glassdesign_tunnel.log')
tl=tunnel_log.open('w')
tunnel=subprocess.Popen([str(cf),'tunnel','--no-autoupdate','--url','http://127.0.0.1:'+port],stdout=tl,stderr=subprocess.STDOUT)
public_url=None
for _ in range(75):
    time.sleep(1)
    match=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',tunnel_log.read_text(errors='replace'))
    if match:
        public_url=match.group(0);break
if not public_url:
    raise RuntimeError('Туннель не запустился; см. /content/glassdesign_tunnel.log')
print('\n✅ GLASSDESIGN ML ОТКРЫТ')
print('Временная ссылка:',public_url)
print('ОСТАВЬТЕ GOOGLE COLAB ВКЛЮЧЕННЫМ, ПОКА ПОЛЬЗУЕТЕСЬ САЙТОМ.')
print('Для повторного запуска в другой день нужно выполнить этот блокнот снова.')
